In [3]:
!pip install pyspark

In [4]:
import pyspark
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, regexp_extract, year

In [5]:
spark = SparkSession.builder.getOrCreate()

In [6]:
df_video = spark.read.csv('/content/videos-stats.csv', header = True, inferSchema = True)
df_video.show()

+---+--------------------+-----------+------------+-------+--------+--------+-----------+
|_c0|               Title|   Video ID|Published At|Keyword|   Likes|Comments|      Views|
+---+--------------------+-----------+------------+-------+--------+--------+-----------+
|  0|Apple Pay Is Kill...|wAZZ-UWGVHI|  2022-08-23|   tech|  3407.0|   672.0|   135612.0|
|  1|The most EXPENSIV...|b3x28s61q3c|  2022-08-24|   tech| 76779.0|  4306.0|  1758063.0|
|  2|My New House Gami...|4mgePWWCAmA|  2022-08-23|   tech| 63825.0|  3338.0|  1564007.0|
|  3|Petrol Vs Liquid ...|kXiYSI7H2b0|  2022-08-23|   tech| 71566.0|  1426.0|   922918.0|
|  4|Best Back to Scho...|ErMwWXQxHp0|  2022-08-08|   tech| 96513.0|  5155.0|  1855644.0|
|  5|Brewmaster Answer...|18fwz9Itbvo|  2021-11-05|   tech| 33570.0|  1643.0|   943119.0|
|  6|Tech Monopolies: ...|jXf04bhcjbg|  2022-06-13|   tech|135047.0|  9367.0|  5937790.0|
|  7|I bought the STRA...|2TqOmtTAMRY|  2022-08-07|   tech|216935.0| 12605.0|  4782514.0|
|  8|15 Em

In [7]:
df_video = df_video.fillna({'Likes': 0, 'Comments': 0, 'Views': 0})
df_video.show()

+---+--------------------+-----------+------------+-------+--------+--------+-----------+
|_c0|               Title|   Video ID|Published At|Keyword|   Likes|Comments|      Views|
+---+--------------------+-----------+------------+-------+--------+--------+-----------+
|  0|Apple Pay Is Kill...|wAZZ-UWGVHI|  2022-08-23|   tech|  3407.0|   672.0|   135612.0|
|  1|The most EXPENSIV...|b3x28s61q3c|  2022-08-24|   tech| 76779.0|  4306.0|  1758063.0|
|  2|My New House Gami...|4mgePWWCAmA|  2022-08-23|   tech| 63825.0|  3338.0|  1564007.0|
|  3|Petrol Vs Liquid ...|kXiYSI7H2b0|  2022-08-23|   tech| 71566.0|  1426.0|   922918.0|
|  4|Best Back to Scho...|ErMwWXQxHp0|  2022-08-08|   tech| 96513.0|  5155.0|  1855644.0|
|  5|Brewmaster Answer...|18fwz9Itbvo|  2021-11-05|   tech| 33570.0|  1643.0|   943119.0|
|  6|Tech Monopolies: ...|jXf04bhcjbg|  2022-06-13|   tech|135047.0|  9367.0|  5937790.0|
|  7|I bought the STRA...|2TqOmtTAMRY|  2022-08-07|   tech|216935.0| 12605.0|  4782514.0|
|  8|15 Em

In [8]:
df_comentario = spark.read.csv('/content/comments.csv', header = True,inferSchema = True)
df_comentario.show()

+--------------+-----------+--------------------+------+---------+
|           _c0|   Video ID|             Comment| Likes|Sentiment|
+--------------+-----------+--------------------+------+---------+
|             0|wAZZ-UWGVHI|Let's not forget ...|  95.0|      1.0|
|             1|wAZZ-UWGVHI|Here in NZ 50% of...|  19.0|      0.0|
|             2|wAZZ-UWGVHI|I will forever ac...| 161.0|      2.0|
|             3|wAZZ-UWGVHI|Whenever I go to ...|   8.0|      0.0|
|             4|wAZZ-UWGVHI|Apple Pay is so c...|  34.0|      2.0|
|             5|wAZZ-UWGVHI|We’ve been houndi...|   8.0|      1.0|
|             6|wAZZ-UWGVHI|We only got Apple...|  29.0|      2.0|
|             7|wAZZ-UWGVHI|For now, I need b...|   7.0|      1.0|
|             8|wAZZ-UWGVHI|In the United Sta...|   2.0|      2.0|
|             9|wAZZ-UWGVHI|In Cambodia, we h...|  28.0|      1.0|
|            10|b3x28s61q3c|Wow, you really w...|1344.0|      2.0|
|            11|b3x28s61q3c|The lab is the mo...| 198.0|      

In [9]:
print(f"Quantidade de registros no video_stats: {df_video.count()}, quantidade de registros no comments: {df_comentario.count()}")

Quantidade de registros no video_stats: 1881, quantidade de registros no comments: 30036


In [10]:
df_video = df_video.filter(col('Video ID').isNotNull())
df_comentario = df_comentario.filter(col('Video ID').isNotNull())

print(f"Quantidade de registros df_video após remoção de nulos em 'Video ID': {df_video.count()}")
print(f"Quantidade de registros df_comentario após remoção de nulos em 'Video ID': {df_comentario.count()}")

Quantidade de registros df_video após remoção de nulos em 'Video ID': 1881
Quantidade de registros df_comentario após remoção de nulos em 'Video ID': 22555


In [11]:
df_video = df_video.dropDuplicates(['Video ID'])
print(f"Quantidade de registros df_video após remoção de duplicados em 'Video ID': {df_video.count()}")

Quantidade de registros df_video após remoção de duplicados em 'Video ID': 1869


In [12]:
df_comentario = (
    df_comentario
    .withColumn("Likes", col("Likes").cast("double").cast("int"))
    .withColumn("Sentiment", col("Sentiment").cast("double").cast("int"))
    .withColumnRenamed("Likes", "Likes Comment")
)

df_comentario.printSchema()
df_comentario.show(5)

root
 |-- _c0: string (nullable = true)
 |-- Video ID: string (nullable = true)
 |-- Comment: string (nullable = true)
 |-- Likes Comment: integer (nullable = true)
 |-- Sentiment: integer (nullable = true)

+---+-----------+--------------------+-------------+---------+
|_c0|   Video ID|             Comment|Likes Comment|Sentiment|
+---+-----------+--------------------+-------------+---------+
|  0|wAZZ-UWGVHI|Let's not forget ...|           95|        1|
|  1|wAZZ-UWGVHI|Here in NZ 50% of...|           19|        0|
|  2|wAZZ-UWGVHI|I will forever ac...|          161|        2|
|  3|wAZZ-UWGVHI|Whenever I go to ...|            8|        0|
|  4|wAZZ-UWGVHI|Apple Pay is so c...|           34|        2|
+---+-----------+--------------------+-------------+---------+
only showing top 5 rows


In [13]:
df_video = df_video.withColumn('Interaction', col('Likes') + col('Comments') + col('Views'))
df_video.show()

+----+--------------------+-----------+------------+----------------+--------+--------+------------+------------+
| _c0|               Title|   Video ID|Published At|         Keyword|   Likes|Comments|       Views| Interaction|
+----+--------------------+-----------+------------+----------------+--------+--------+------------+------------+
| 986|ASMR MUKBANG DOUB...|--ZI0dSbbNU|  2020-04-18|         mukbang|378858.0| 18860.0| 1.7975269E7| 1.8372987E7|
|  71|Deadly car bomb d...|--hxd1CrOqg|  2022-08-22|            news|  6379.0|  4853.0|    808787.0|    820019.0|
|  48|How Biden&#39;s s...|--ixiTypG8g|  2022-08-24|            news|  1029.0|  2347.0|     97434.0|    100810.0|
| 993|Celebrating My 40...|-64r1hcxtV4|  2022-05-30|         mukbang| 45628.0| 17264.0|   5283664.0|   5346556.0|
|1456|Physics Review - ...|-6IgkG5yZfo|  2017-01-02|         physics| 10959.0|   525.0|    844015.0|    855499.0|
| 949|Eating ONLY KOREA...|-7hzaGya86g|  2022-07-17|            food| 18957.0|   566.0| 

In [14]:
df_video = df_video.withColumn('Published At', col('Published At').cast('date'))
df_video.show()

+----+--------------------+-----------+------------+----------------+--------+--------+------------+------------+
| _c0|               Title|   Video ID|Published At|         Keyword|   Likes|Comments|       Views| Interaction|
+----+--------------------+-----------+------------+----------------+--------+--------+------------+------------+
| 986|ASMR MUKBANG DOUB...|--ZI0dSbbNU|  2020-04-18|         mukbang|378858.0| 18860.0| 1.7975269E7| 1.8372987E7|
|  71|Deadly car bomb d...|--hxd1CrOqg|  2022-08-22|            news|  6379.0|  4853.0|    808787.0|    820019.0|
|  48|How Biden&#39;s s...|--ixiTypG8g|  2022-08-24|            news|  1029.0|  2347.0|     97434.0|    100810.0|
| 993|Celebrating My 40...|-64r1hcxtV4|  2022-05-30|         mukbang| 45628.0| 17264.0|   5283664.0|   5346556.0|
|1456|Physics Review - ...|-6IgkG5yZfo|  2017-01-02|         physics| 10959.0|   525.0|    844015.0|    855499.0|
| 949|Eating ONLY KOREA...|-7hzaGya86g|  2022-07-17|            food| 18957.0|   566.0| 

In [15]:
df_video = df_video.withColumn('Year', year(col('Published At')))
df_video.show()

+----+--------------------+-----------+------------+----------------+--------+--------+------------+------------+----+
| _c0|               Title|   Video ID|Published At|         Keyword|   Likes|Comments|       Views| Interaction|Year|
+----+--------------------+-----------+------------+----------------+--------+--------+------------+------------+----+
| 986|ASMR MUKBANG DOUB...|--ZI0dSbbNU|  2020-04-18|         mukbang|378858.0| 18860.0| 1.7975269E7| 1.8372987E7|2020|
|  71|Deadly car bomb d...|--hxd1CrOqg|  2022-08-22|            news|  6379.0|  4853.0|    808787.0|    820019.0|2022|
|  48|How Biden&#39;s s...|--ixiTypG8g|  2022-08-24|            news|  1029.0|  2347.0|     97434.0|    100810.0|2022|
| 993|Celebrating My 40...|-64r1hcxtV4|  2022-05-30|         mukbang| 45628.0| 17264.0|   5283664.0|   5346556.0|2022|
|1456|Physics Review - ...|-6IgkG5yZfo|  2017-01-02|         physics| 10959.0|   525.0|    844015.0|    855499.0|2017|
| 949|Eating ONLY KOREA...|-7hzaGya86g|  2022-07

In [16]:
df_join_video_comments = df_video.join(df_comentario, on='Video ID', how='inner')
df_join_video_comments.show()

+-----------+---+--------------------+------------+-------+-------+--------+---------+-----------+----+---+--------------------+-------------+---------+
|   Video ID|_c0|               Title|Published At|Keyword|  Likes|Comments|    Views|Interaction|Year|_c0|             Comment|Likes Comment|Sentiment|
+-----------+---+--------------------+------------+-------+-------+--------+---------+-----------+----+---+--------------------+-------------+---------+
|wAZZ-UWGVHI|  0|Apple Pay Is Kill...|  2022-08-23|   tech| 3407.0|   672.0| 135612.0|   139691.0|2022|  0|Let's not forget ...|           95|        1|
|wAZZ-UWGVHI|  0|Apple Pay Is Kill...|  2022-08-23|   tech| 3407.0|   672.0| 135612.0|   139691.0|2022|  1|Here in NZ 50% of...|           19|        0|
|wAZZ-UWGVHI|  0|Apple Pay Is Kill...|  2022-08-23|   tech| 3407.0|   672.0| 135612.0|   139691.0|2022|  2|I will forever ac...|          161|        2|
|wAZZ-UWGVHI|  0|Apple Pay Is Kill...|  2022-08-23|   tech| 3407.0|   672.0| 13561

In [17]:
df_join_video_comments = df_join_video_comments.drop('_c0')
df_join_video_comments.show()

+-----------+--------------------+------------+-------+-------+--------+---------+-----------+----+--------------------+-------------+---------+
|   Video ID|               Title|Published At|Keyword|  Likes|Comments|    Views|Interaction|Year|             Comment|Likes Comment|Sentiment|
+-----------+--------------------+------------+-------+-------+--------+---------+-----------+----+--------------------+-------------+---------+
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407.0|   672.0| 135612.0|   139691.0|2022|Let's not forget ...|           95|        1|
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407.0|   672.0| 135612.0|   139691.0|2022|Here in NZ 50% of...|           19|        0|
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407.0|   672.0| 135612.0|   139691.0|2022|I will forever ac...|          161|        2|
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407.0|   672.0| 135612.0|   139691.0|2022|Whenever I go to ...|          

In [22]:
df_join_video_comments.write.mode('overwrite').parquet('videos-comments-tratados-parquet')

{"ts": "2026-08-10 23:50:29.267", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[CAST_INVALID_INPUT] The value ' and his  team' of the type \"STRING\" cannot be cast to \"DOUBLE\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018", "context": {"file": "line 3 in cell [12]", "line": "", "fragment": "cast", "errorClass": "CAST_INVALID_INPUT"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o135.parquet.\n: org.apache.spark.SparkNumberFormatException: [CAST_INVALID_INPUT] The value ' and his  team' of the type \"STRING\" cannot be cast to \"DOUBLE\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018\n== DataFrame ==\n\"cast\" was called from\nline 3 in cell [12]\n\n\tat org.apache.spark.sql.erro

NumberFormatException: [CAST_INVALID_INPUT] The value ' and his  team' of the type "STRING" cannot be cast to "DOUBLE" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"cast" was called from
line 3 in cell [12]


In [19]:
df_join_video_comments = df_join_video_comments.drop('_c0')
df_join_video_comments.show()

+-----------+--------------------+------------+-------+-------+--------+---------+-----------+----+--------------------+-------------+---------+
|   Video ID|               Title|Published At|Keyword|  Likes|Comments|    Views|Interaction|Year|             Comment|Likes Comment|Sentiment|
+-----------+--------------------+------------+-------+-------+--------+---------+-----------+----+--------------------+-------------+---------+
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407.0|   672.0| 135612.0|   139691.0|2022|Let's not forget ...|           95|        1|
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407.0|   672.0| 135612.0|   139691.0|2022|Here in NZ 50% of...|           19|        0|
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407.0|   672.0| 135612.0|   139691.0|2022|I will forever ac...|          161|        2|
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407.0|   672.0| 135612.0|   139691.0|2022|Whenever I go to ...|          

In [23]:
df_join_video_comments.write.mode('overwrite').parquet('videos-comments-tratados-parquet')

{"ts": "2026-08-10 23:50:34.480", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[CAST_INVALID_INPUT] The value ' and his  team' of the type \"STRING\" cannot be cast to \"DOUBLE\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018", "context": {"file": "line 3 in cell [12]", "line": "", "fragment": "cast", "errorClass": "CAST_INVALID_INPUT"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o145.parquet.\n: org.apache.spark.SparkNumberFormatException: [CAST_INVALID_INPUT] The value ' and his  team' of the type \"STRING\" cannot be cast to \"DOUBLE\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018\n== DataFrame ==\n\"cast\" was called from\nline 3 in cell [12]\n\n\tat org.apache.spark.sql.erro

NumberFormatException: [CAST_INVALID_INPUT] The value ' and his  team' of the type "STRING" cannot be cast to "DOUBLE" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"cast" was called from
line 3 in cell [12]


In [24]:
df_us_videos = spark.read.csv('/content/USvideos.csv', header = True, inferSchema = True)
df_us_videos.show()

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/content/USvideos.csv. SQLSTATE: 42K03

In [25]:
df_join_video_usvideos = df_video.join(df_us_videos, on='Title', how='inner')
df_join_video_usvideos.show()

NameError: name 'df_us_videos' is not defined

In [26]:
df_video.select([count(when(col(c).isNull(), c)).alias(c) for c in df_video.columns]).show()

+---+-----+--------+------------+-------+-----+--------+-----+-----------+----+
|_c0|Title|Video ID|Published At|Keyword|Likes|Comments|Views|Interaction|Year|
+---+-----+--------+------------+-------+-----+--------+-----+-----------+----+
|  0|    0|       0|           0|      0|    0|       0|    0|          0|   0|
+---+-----+--------+------------+-------+-----+--------+-----+-----------+----+



In [27]:
df_video = df_video.drop('_c0')
df_video.show()
df_video.write.mode('overwrite').parquet('videos-tratados-parquet')

+--------------------+-----------+------------+----------------+--------+--------+------------+------------+----+
|               Title|   Video ID|Published At|         Keyword|   Likes|Comments|       Views| Interaction|Year|
+--------------------+-----------+------------+----------------+--------+--------+------------+------------+----+
|ASMR MUKBANG DOUB...|--ZI0dSbbNU|  2020-04-18|         mukbang|378858.0| 18860.0| 1.7975269E7| 1.8372987E7|2020|
|Deadly car bomb d...|--hxd1CrOqg|  2022-08-22|            news|  6379.0|  4853.0|    808787.0|    820019.0|2022|
|How Biden&#39;s s...|--ixiTypG8g|  2022-08-24|            news|  1029.0|  2347.0|     97434.0|    100810.0|2022|
|Celebrating My 40...|-64r1hcxtV4|  2022-05-30|         mukbang| 45628.0| 17264.0|   5283664.0|   5346556.0|2022|
|Physics Review - ...|-6IgkG5yZfo|  2017-01-02|         physics| 10959.0|   525.0|    844015.0|    855499.0|2017|
|Eating ONLY KOREA...|-7hzaGya86g|  2022-07-17|            food| 18957.0|   566.0|    70

In [28]:
df_join_video_comments = df_join_video_comments.drop('_c0')
df_join_video_comments.show()
df_join_video_comments.write.mode('overwrite').parquet('videos-comments-tratados-parquet')

+-----------+--------------------+------------+-------+-------+--------+---------+-----------+----+--------------------+-------------+---------+
|   Video ID|               Title|Published At|Keyword|  Likes|Comments|    Views|Interaction|Year|             Comment|Likes Comment|Sentiment|
+-----------+--------------------+------------+-------+-------+--------+---------+-----------+----+--------------------+-------------+---------+
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407.0|   672.0| 135612.0|   139691.0|2022|Let's not forget ...|           95|        1|
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407.0|   672.0| 135612.0|   139691.0|2022|Here in NZ 50% of...|           19|        0|
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407.0|   672.0| 135612.0|   139691.0|2022|I will forever ac...|          161|        2|
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407.0|   672.0| 135612.0|   139691.0|2022|Whenever I go to ...|          

{"ts": "2026-08-10 23:50:58.673", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[CAST_INVALID_INPUT] The value ' and his  team' of the type \"STRING\" cannot be cast to \"DOUBLE\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018", "context": {"file": "line 3 in cell [12]", "line": "", "fragment": "cast", "errorClass": "CAST_INVALID_INPUT"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o256.parquet.\n: org.apache.spark.SparkNumberFormatException: [CAST_INVALID_INPUT] The value ' and his  team' of the type \"STRING\" cannot be cast to \"DOUBLE\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018\n== DataFrame ==\n\"cast\" was called from\nline 3 in cell [12]\n\n\tat org.apache.spark.sql.erro

NumberFormatException: [CAST_INVALID_INPUT] The value ' and his  team' of the type "STRING" cannot be cast to "DOUBLE" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"cast" was called from
line 3 in cell [12]
